# 1. 파일명 정리

In [13]:
from pathlib import Path
import shutil
import os

raw_dir = Path(r"C:\gachikium\dataset\dataset_v2_raw")
renamed_dir = Path(r"C:\gachikium\dataset\dataset_v2_renamed")

print(raw_dir.exists())
print(raw_dir)
print(renamed_dir)

True
C:\gachikium\dataset\dataset_v2_raw
C:\gachikium\dataset\dataset_v2_renamed


# 수량 확인

In [1]:
from pathlib import Path

data_dir = Path(r"C:\gachikium\dataset\dataset_v3_merged_cropped")

for class_dir in data_dir.iterdir():
    if class_dir.is_dir():
        count = len(list(class_dir.glob("*")))
        print(class_dir.name, count)

강아지상 50
고양이상 50
곰상 50
꼬부기상 50
도롱뇽상 50
사슴상 50
여우상 50
쥐상 50
쿼카상 50
토끼상 50


merged + cropped dataset 완성
→ 이걸 dataset_v3로 선언
→ 클래스별 개수 확인
→ split 다시 생성
→ 새 노트북에서 ResNet+Optuna 다시 실행
→ baseline과 비교

# split

In [3]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [8]:
import random
import shutil
from pathlib import Path

# 원본 데이터
SRC_DIR = Path(r"C:\gachikium\dataset\dataset_v3_merged_cropped")

# 새로 만들 폴더
TRAIN_DIR = Path(r"C:\gachikium\dataset\train_resnet_2")
TEST_DIR  = Path(r"C:\gachikium\dataset\test_resnet_2")

TRAIN_RATIO = 0.8
SEED = 42
CLEAR_OUTPUT = True

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".jfif"}

random.seed(SEED)

# 폴더 초기화
def clear_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

# 이미지 리스트
def list_images(folder: Path):
    return [p for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in IMG_EXTS]

# 안전 복사
def safe_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

# train/test split
def split_train_test(paths, train_ratio=0.8):
    paths = list(paths)
    random.shuffle(paths)
    n = len(paths)
    n_train = int(n * train_ratio)
    return paths[:n_train], paths[n_train:]

# 폴더 존재 확인
if not SRC_DIR.exists():
    raise FileNotFoundError(f"Not found: {SRC_DIR}")

# output 초기화
if CLEAR_OUTPUT:
    clear_dir(TRAIN_DIR)
    clear_dir(TEST_DIR)

# 클래스별 split
for cls_dir in sorted([d for d in SRC_DIR.iterdir() if d.is_dir()], key=lambda x: x.name):

    imgs = list_images(cls_dir)

    if len(imgs) != 50:
        print(f"[WARN] {cls_dir.name}: expected 50 but got {len(imgs)}")

    train_list, test_list = split_train_test(imgs, TRAIN_RATIO)

    for src in train_list:
        safe_copy(src, TRAIN_DIR / cls_dir.name / src.name)

    for src in test_list:
        safe_copy(src, TEST_DIR / cls_dir.name / src.name)

    print(f"{cls_dir.name}: total={len(imgs)}, train={len(train_list)}, test={len(test_list)}")

print("\nDONE")
print("train_resnet_2:", TRAIN_DIR)
print("test_resnet_2 :", TEST_DIR)

강아지상: total=50, train=40, test=10
고양이상: total=50, train=40, test=10
곰상: total=50, train=40, test=10
꼬부기상: total=50, train=40, test=10
도롱뇽상: total=50, train=40, test=10
사슴상: total=50, train=40, test=10
여우상: total=50, train=40, test=10
쥐상: total=50, train=40, test=10
쿼카상: total=50, train=40, test=10
토끼상: total=50, train=40, test=10

DONE
train_resnet_2: C:\gachikium\dataset\train_resnet_2
test_resnet_2 : C:\gachikium\dataset\test_resnet_2


# split 이 잘 됐는지 확인 코드

In [10]:
from pathlib import Path

train_path = Path(r"C:\gachikium\dataset\train_resnet_2")
test_path  = Path(r"C:\gachikium\dataset\test_resnet_2")

print("train_resnet_2 exists:", train_path.exists())
print("test_resnet_2 exists :", test_path.exists())

print("Train classes:")
print([d.name for d in train_path.iterdir() if d.is_dir()])

print("\nTest classes:")
print([d.name for d in test_path.iterdir() if d.is_dir()])

train_resnet_2 exists: True
test_resnet_2 exists : True
Train classes:
['강아지상', '고양이상', '곰상', '꼬부기상', '도롱뇽상', '사슴상', '여우상', '쥐상', '쿼카상', '토끼상']

Test classes:
['강아지상', '고양이상', '곰상', '꼬부기상', '도롱뇽상', '사슴상', '여우상', '쥐상', '쿼카상', '토끼상']


In [11]:
import os

for cls in os.listdir(train_path):
    cls_path = train_path / cls
    if cls_path.is_dir():
        print(cls, "train:", len(os.listdir(cls_path)))

print("------")

for cls in os.listdir(test_path):
    cls_path = test_path / cls
    if cls_path.is_dir():
        print(cls, "test:", len(os.listdir(cls_path)))

강아지상 train: 40
고양이상 train: 40
곰상 train: 40
꼬부기상 train: 40
도롱뇽상 train: 40
사슴상 train: 40
여우상 train: 40
쥐상 train: 40
쿼카상 train: 40
토끼상 train: 40
------
강아지상 test: 10
고양이상 test: 10
곰상 test: 10
꼬부기상 test: 10
도롱뇽상 test: 10
사슴상 test: 10
여우상 test: 10
쥐상 test: 10
쿼카상 test: 10
토끼상 test: 10


# Dataloader
- train_resnet2 안에 데이터는 train/val로 나누고
- test_resnet2는 최종 test로 그대로 쓰면 됨

In [12]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch

train_dataset_full = ImageFolder(
    r"C:\gachikium\dataset\train_resnet_2",
    transform=transform_train
)

test_dataset = ImageFolder(
    r"C:\gachikium\dataset\test_resnet_2",
    transform=transform_test
)

class_names = train_dataset_full.classes
print(class_names)
print("train full size:", len(train_dataset_full))
print("test size:", len(test_dataset))

# train / val 분리
train_size = int(len(train_dataset_full) * 0.8)
val_size = len(train_dataset_full) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=generator
)

print("train split size:", len(train_dataset))
print("val split size:", len(val_dataset))

['강아지상', '고양이상', '곰상', '꼬부기상', '도롱뇽상', '사슴상', '여우상', '쥐상', '쿼카상', '토끼상']
train full size: 400
test size: 100
train split size: 320
val split size: 80


In [13]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch

train_dataset_full = ImageFolder(
    r"C:\gachikium\dataset\train_resnet_2",
    transform=transform_train
)

test_dataset = ImageFolder(
    r"C:\gachikium\dataset\test_resnet_2",
    transform=transform_test
)

class_names = train_dataset_full.classes
print(class_names)
print("train full size:", len(train_dataset_full))
print("test size:", len(test_dataset))

# train / val 분리
train_size = int(len(train_dataset_full) * 0.8)
val_size = len(train_dataset_full) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=generator
)

print("train split size:", len(train_dataset))
print("val split size:", len(val_dataset))

['강아지상', '고양이상', '곰상', '꼬부기상', '도롱뇽상', '사슴상', '여우상', '쥐상', '쿼카상', '토끼상']
train full size: 400
test size: 100
train split size: 320
val split size: 80


# Optuna + ResNet18 튜닝 코드 

In [20]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = len(class_names)

def objective(trial):

    # 탐색할 하이퍼파라미터
    batch_size = trial.suggest_categorical("batch_size",[8,16,32,64])

    lr = trial.suggest_float(
    "lr",
    1e-5,
    1e-3,
    log=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    # ResNet18 불러오기
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # classifier 교체
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    num_epochs = 8
    best_val_loss = float("inf")

    for epoch in range(num_epochs):

        # -----------------
        # train
        # -----------------

        model.train()
        train_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # -----------------
        # validation
        # -----------------

        model.eval()

        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                val_loss += loss.item()

                preds = outputs.argmax(dim=1)

                correct += (preds == labels).sum().item()

                total += labels.size(0)

        val_loss /= len(val_loader)

        val_acc = correct / total

        print(
            f"[Trial {trial.number}] "
            f"Epoch {epoch+1}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f}"
        )

        trial.report(val_loss, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_loss < best_val_loss:
            best_val_loss = val_loss

    return best_val_loss

# optuna 실행 셀

In [21]:
study = optuna.create_study(direction="minimize")

study.optimize(objective, n_trials=60)

print("Best trial:")
print("  value:", study.best_trial.value)
print("  params:", study.best_trial.params)

best_params = study.best_trial.params
best_batch_size = best_params["batch_size"]
best_lr = best_params["lr"]

print("Final batch_size:", best_batch_size)
print("Final lr:", best_lr)

[I 2026-03-10 16:47:23,282] A new study created in memory with name: no-name-292e158e-8bc3-4e4c-909e-0429cb85d859


[Trial 0] Epoch 1/8 | train_loss=2.3091 | val_loss=2.3778 | val_acc=0.1000
[Trial 0] Epoch 2/8 | train_loss=1.7029 | val_loss=2.1989 | val_acc=0.2875
[Trial 0] Epoch 3/8 | train_loss=1.3157 | val_loss=2.0087 | val_acc=0.4250
[Trial 0] Epoch 4/8 | train_loss=1.0221 | val_loss=1.8139 | val_acc=0.5000
[Trial 0] Epoch 5/8 | train_loss=0.8076 | val_loss=1.6203 | val_acc=0.5500
[Trial 0] Epoch 6/8 | train_loss=0.6170 | val_loss=1.4583 | val_acc=0.6000
[Trial 0] Epoch 7/8 | train_loss=0.4910 | val_loss=1.3244 | val_acc=0.6750


[I 2026-03-10 16:47:51,391] Trial 0 finished with value: 1.2008774280548096 and parameters: {'batch_size': 64, 'lr': 3.170362766150648e-05}. Best is trial 0 with value: 1.2008774280548096.


[Trial 0] Epoch 8/8 | train_loss=0.3878 | val_loss=1.2009 | val_acc=0.7250
[Trial 1] Epoch 1/8 | train_loss=1.4006 | val_loss=0.9693 | val_acc=0.6875
[Trial 1] Epoch 2/8 | train_loss=0.4467 | val_loss=0.6512 | val_acc=0.8375
[Trial 1] Epoch 3/8 | train_loss=0.1933 | val_loss=0.5138 | val_acc=0.8375
[Trial 1] Epoch 4/8 | train_loss=0.1179 | val_loss=0.4461 | val_acc=0.8625
[Trial 1] Epoch 5/8 | train_loss=0.0813 | val_loss=0.6205 | val_acc=0.8500
[Trial 1] Epoch 6/8 | train_loss=0.0880 | val_loss=0.7245 | val_acc=0.8000
[Trial 1] Epoch 7/8 | train_loss=0.0806 | val_loss=0.6995 | val_acc=0.7625


[I 2026-03-10 16:48:08,823] Trial 1 finished with value: 0.4460617661476135 and parameters: {'batch_size': 8, 'lr': 0.00021069722480540002}. Best is trial 1 with value: 0.4460617661476135.


...

[Trial 57] Epoch 1/8 | train_loss=1.4501 | val_loss=1.1090 | val_acc=0.6375


[I 2026-03-10 16:54:36,816] Trial 58 pruned. 


[Trial 58] Epoch 1/8 | train_loss=1.4927 | val_loss=1.0753 | val_acc=0.5750


[I 2026-03-10 16:54:38,885] Trial 59 pruned. 


[Trial 59] Epoch 1/8 | train_loss=1.3942 | val_loss=1.6041 | val_acc=0.5375
Best trial:
  value: 0.30117387473583224
  params: {'batch_size': 8, 'lr': 0.00016865884467300648}
Final batch_size: 8
Final lr: 0.00016865884467300648
